# Intermediate feature perturbation study

The assignment PDF labels this item as **[Optional]** on page 2, but this notebook lets you run it if you want it in the report.

In [1]:

from assignment2_gpu_split_module import *
import pandas as pd

IMAGENET100_MODE = "local_existing"
IMAGENET100_ROOT = ROOT / "data" / "imagenet100"
TRAIN_PRETRAINED = False

# Choose one or more representative pairs.
PAIR_LIST = [
    ("cifar10", "resnet"),
    ("fashion_mnist", "resnet"),
    ("imagenet100", "resnet"),
]


2026-03-21 21:27:45.461472: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-21 21:27:45.543366: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Project root: /data/Sajjan_Singh/cv_assign_04/cv_assign_04_gpu_split_local_fast5/cv_assign_04_gpu_split_local_fast5/cv_assign_04
Device: cuda
PyTorch version: 2.6.0+cu124
CUDA available: True
GPU name: NVIDIA H100 NVL
cifar10              -> https://www.cs.toronto.edu/~kriz/cifar.html
fashion_mnist        -> https://github.com/zalandoresearch/fashion-mnist
imagenet100_hf       -> https://huggingface.co/datasets/clane9/imagenet-100
imagenet100_kaggle   -> https://www.kaggle.com/datasets/ambityga/imagenet100
imagecorruptions     -> https://github.com/bethgelab/imagecorruptions
robustness_repo      -> https://github.com/hendrycks/robustness


- input size: `224`
- policy K values: `1, 3, 5, 10, 15`
- backbones: `vgg16_bn, resnet50, convnext_tiny, vit_base_patch16_224`

In [ ]:
all_feat_rows = []

# Iterate through every model/dataset combo defined in PAIR_LIST
for dataset_name, model_name in PAIR_LIST:
    print("\n" + "="*100)
    print(f"Feature perturbation study: {dataset_name} + {model_name}")
    print("="*100)
    
    # Run the actual ablation experiment (modifying features to see impact)
    df = run_feature_perturbation_ablation(
        dataset_name=dataset_name,
        model_name=model_name,
        imagenet100_source=IMAGENET100_MODE,
        imagenet100_root=IMAGENET100_ROOT,
        train_pretrained=TRAIN_PRETRAINED,
    )
    
    # Tag the resulting dataframe so we know which run it belongs to
    df["dataset"] = dataset_name
    df["model"] = model_name
    all_feat_rows.append(df)

# Merge all individual experiment DFs into one giant master table
feature_perturbation_master = pd.concat(all_feat_rows, ignore_index=True)

# Save results to CSV so we don't have to re-run this 10-hour loop later
feature_perturbation_master.to_csv(ROOT / "tables" / "feature_perturbation_master.csv", index=False)

# Show the full table and print the save path for easy access
display(feature_perturbation_master)
print(ROOT / "tables" / "feature_perturbation_master.csv")


Feature perturbation study: cifar10 + resnet
torch.compile enabled for this run.
Epoch 1/8 | training...
Epoch 1/8 | clean validation...
Epoch 1/8 | corrupted validation over 5 corruption types...
  validating corruption: gaussian_noise
  validating corruption: motion_blur
  validating corruption: fog
  validating corruption: brightness
  validating corruption: jpeg_compression
{'epoch': 1, 'train_loss': 2.176234164428711, 'train_acc': 0.2246, 'lr': 0.009619397662556433, 'time_sec': 88.95284581184387, 'val_acc__clean': 0.2845, 'val_acc__gaussian_noise': 0.2594, 'val_acc__motion_blur': 0.2809, 'val_acc__fog': 0.1744, 'val_acc__brightness': 0.2563, 'val_acc__jpeg_compression': 0.2848, 'val_acc__corr_k1_s2': 0.2594, 'val_acc__corr_k3_s2': 0.23823333333333332, 'val_acc__corr_k5_s2': 0.25116}
Epoch 2/8 | training...
Epoch 2/8 | clean validation...
Epoch 2/8 | corrupted validation over 5 corruption types...
  validating corruption: gaussian_noise
  validating corruption: motion_blur
  valid

,dataset,model,selector,selector_type,best_epoch,best_val_score,num_val_corruptions,val_corruption_names,val_corruption_severity,test_clean_acc,test_corrupt_mean_acc,robustness_gap,experiment_dir,checkpoint_path,where
0,cifar10,resnet,corr_k3_s2,corrupt,8,0.304133,3,"gaussian_noise, motion_blur, fog",2,0.4789,0.35746,0.12144,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,early
1,cifar10,resnet,corr_k3_s2,corrupt,8,0.298267,3,"gaussian_noise, motion_blur, fog",2,0.4808,0.35476,0.12604,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,mid
2,cifar10,resnet,corr_k5_s2,corrupt,8,0.353300,5,"gaussian_noise, motion_blur, fog, brightness, ...",2,0.4700,0.35580,0.11420,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,late


Saved bonus ablation to: /data/Sajjan_Singh/cv_assign_04/cv_assign_04_gpu_split_local_fast5/cv_assign_04_gpu_split_local_fast5/cv_assign_04/tables/feature_perturbation_cifar10_resnet.csv

Feature perturbation study: fashion_mnist + resnet
torch.compile enabled for this run.
Epoch 1/8 | training...
Epoch 1/8 | clean validation...


W0321 22:18:04.971000 401356 site-packages/torch/_dynamo/convert_frame.py:906] [0/8] torch._dynamo hit config.cache_size_limit (8)
W0321 22:18:04.971000 401356 site-packages/torch/_dynamo/convert_frame.py:906] [0/8]    function: 'forward' (/home/sajjan/.conda/envs/myenv/lib/python3.10/site-packages/timm/models/resnet.py:773)
W0321 22:18:04.971000 401356 site-packages/torch/_dynamo/convert_frame.py:906] [0/8]    last reason: 0/5: Cache line invalidated because L['self']._modules['fc']._forward_hooks[list(L['self']._modules['fc']._forward_hooks.keys())[0]] got deallocated
W0321 22:18:04.971000 401356 site-packages/torch/_dynamo/convert_frame.py:906] [0/8] To log all recompilation reasons, use TORCH_LOGS="recompiles".
W0321 22:18:04.971000 401356 site-packages/torch/_dynamo/convert_frame.py:906] [0/8] To diagnose recompilation issues, see https://pytorch.org/docs/main/torch.compiler_troubleshooting.html.


Epoch 1/8 | corrupted validation over 5 corruption types...
  validating corruption: gaussian_noise
  validating corruption: motion_blur
  validating corruption: fog
  validating corruption: brightness
  validating corruption: jpeg_compression
{'epoch': 1, 'train_loss': 1.553129222869873, 'train_acc': 0.418, 'lr': 0.009619397662556433, 'time_sec': 35.871891498565674, 'val_acc__clean': 0.22808333333333333, 'val_acc__gaussian_noise': 0.11008333333333334, 'val_acc__motion_blur': 0.22116666666666668, 'val_acc__fog': 0.10625, 'val_acc__brightness': 0.12933333333333333, 'val_acc__jpeg_compression': 0.20733333333333334, 'val_acc__corr_k1_s2': 0.11008333333333334, 'val_acc__corr_k3_s2': 0.14583333333333334, 'val_acc__corr_k5_s2': 0.15483333333333335}
Epoch 2/8 | training...
Epoch 2/8 | clean validation...
Epoch 2/8 | corrupted validation over 5 corruption types...
  validating corruption: gaussian_noise
  validating corruption: motion_blur
  validating corruption: fog
  validating corruption: 

...

Saved bonus ablation to: /data/Sajjan_Singh/cv_assign_04/cv_assign_04_gpu_split_local_fast5/cv_assign_04_gpu_split_local_fast5/cv_assign_04/tables/feature_perturbation_fashion_mnist_resnet.csv

Feature perturbation study: imagenet100 + resnet
torch.compile enabled for this run.
Epoch 1/6 | training...
Epoch 1/6 | clean validation...
Epoch 1/6 | corrupted validation over 5 corruption types...
  validating corruption: gaussian_noise
  validating corruption: motion_blur
  validating corruption: fog
  validating corruption: brightness
  validating corruption: jpeg_compression
{'epoch': 1, 'train_loss': 4.281545966221736, 'train_acc': 0.0605982905982906, 'lr': 0.009330127018922194, 'time_sec': 36.699721813201904, 'val_acc__clean': 0.07252136752136752, 'val_acc__gaussian_noise': 0.07303418803418803, 'val_acc__motion_blur': 0.05448717948717949, 'val_acc__fog': 0.018632478632478633, 'val_acc__brightness': 0.05188034188034188, 'val_acc__jpeg_compression': 0.07243589743589744, 'val_acc__corr_k1_

,dataset,model,selector,selector_type,best_epoch,best_val_score,num_val_corruptions,val_corruption_names,val_corruption_severity,test_clean_acc,test_corrupt_mean_acc,robustness_gap,experiment_dir,checkpoint_path,where
0,imagenet100,resnet,corr_k3_s2,corrupt,6,0.083219,3,"gaussian_noise, motion_blur, fog",2,0.198308,0.114954,0.083354,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,early
1,imagenet100,resnet,corr_k3_s2,corrupt,6,0.083262,3,"gaussian_noise, motion_blur, fog",2,0.199000,0.114723,0.084277,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,mid
2,imagenet100,resnet,corr_k3_s2,corrupt,6,0.082721,3,"gaussian_noise, motion_blur, fog",2,0.199077,0.114277,0.084800,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,late


Saved bonus ablation to: /data/Sajjan_Singh/cv_assign_04/cv_assign_04_gpu_split_local_fast5/cv_assign_04_gpu_split_local_fast5/cv_assign_04/tables/feature_perturbation_imagenet100_resnet.csv


,dataset,model,selector,selector_type,best_epoch,best_val_score,num_val_corruptions,val_corruption_names,val_corruption_severity,test_clean_acc,test_corrupt_mean_acc,robustness_gap,experiment_dir,checkpoint_path,where
0,cifar10,resnet,corr_k3_s2,corrupt,8,0.304133,3,"gaussian_noise, motion_blur, fog",2,0.478900,0.357460,0.121440,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,early
1,cifar10,resnet,corr_k3_s2,corrupt,8,0.298267,3,"gaussian_noise, motion_blur, fog",2,0.480800,0.354760,0.126040,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,mid
2,cifar10,resnet,corr_k5_s2,corrupt,8,0.353300,5,"gaussian_noise, motion_blur, fog, brightness, ...",2,0.470000,0.355800,0.114200,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,late
3,fashion_mnist,resnet,corr_k3_s2,corrupt,4,0.341389,3,"gaussian_noise, motion_blur, fog",2,0.557200,0.383380,0.173820,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,early
4,fashion_mnist,resnet,corr_k3_s2,corrupt,6,0.405750,3,"gaussian_noise, motion_blur, fog",2,0.775200,0.443280,0.331920,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,mid
5,fashion_mnist,resnet,corr_k3_s2,corrupt,6,0.405389,3,"gaussian_noise, motion_blur, fog",2,0.776400,0.442900,0.333500,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,late
6,imagenet100,resnet,corr_k3_s2,corrupt,6,0.083219,3,"gaussian_noise, motion_blur, fog",2,0.198308,0.114954,0.083354,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,early
7,imagenet100,resnet,corr_k3_s2,corrupt,6,0.083262,3,"gaussian_noise, motion_blur, fog",2,0.199000,0.114723,0.084277,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,mid
8,imagenet100,resnet,corr_k3_s2,corrupt,6,0.082721,3,"gaussian_noise, motion_blur, fog",2,0.199077,0.114277,0.084800,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,late


/data/Sajjan_Singh/cv_assign_04/cv_assign_04_gpu_split_local_fast5/cv_assign_04_gpu_split_local_fast5/cv_assign_04/tables/feature_perturbation_master.csv
